In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import torch
from torch import Tensor

# ── dimensions ────────────────────────────────────────────────────────────────
BATCH     = 4
SEQ_LEN   = 512
DIM       = 768
N_GROUPS1 = 50   # ~words in modality 1
N_GROUPS2 = 40   # ~words in modality 2

# ── attributions: [batch, seq_len, dim] → summed to [batch, seq_len] ─────────
attr1 = torch.randn(BATCH, SEQ_LEN, DIM)
attr2 = torch.randn(BATCH, SEQ_LEN, DIM)

attributions = (attr1, attr2)

# ── feature masks: randomly assign each token to a group ──────────────────────
# shape must match attributions: [batch, seq_len]
# each value is a group id in [0, N_GROUPS - 1]
def make_feature_mask(batch, seq_len, n_groups):
    """Randomly assign tokens to groups, ensuring every group gets at least 1 token."""
    mask = torch.zeros(batch, seq_len, dtype=torch.long)
    for b in range(batch):
        # guarantee coverage: place one token per group first
        base = torch.arange(n_groups)                           # [n_groups]
        rest = torch.randint(0, n_groups, (seq_len - n_groups,))
        perm = torch.cat([base, rest])[torch.randperm(seq_len)]
        mask[b] = perm
    return mask

mask1 = make_feature_mask(BATCH, SEQ_LEN, N_GROUPS1).unsqueeze(-1).expand_as(attr1)  # [4, 512]
mask2 = make_feature_mask(BATCH, SEQ_LEN, N_GROUPS2).unsqueeze(-1).expand_as(attr2) + N_GROUPS1  # [4, 512]

feature_mask = (mask1, mask2)

# ── quick sanity checks ───────────────────────────────────────────────────────
print("attr1 shape     :", attr1.shape)          # [4, 512]
print("attr2 shape     :", attr2.shape)          # [4, 512]
print("mask1 shape     :", mask1.shape)          # [4, 512]
print("mask2 shape     :", mask2.shape)          # [4, 512]
print("mask1 unique[0] :", mask1[0].unique().shape[0], "groups")  # 50
print("mask2 unique[0] :", mask2[0].unique().shape[0], "groups")  # 40

attr1 shape     : torch.Size([4, 512, 768])
attr2 shape     : torch.Size([4, 512, 768])
mask1 shape     : torch.Size([4, 512, 768])
mask2 shape     : torch.Size([4, 512, 768])
mask1 unique[0] : 50 groups
mask2 unique[0] : 40 groups


In [28]:
"""
Modality Top-K Fraction metric.

Given attributions as a tuple of tensors (one per modality / feature group),
measures which modality contributes the most important features by counting
membership in the global top-k.

Input:
    attributions: tuple[Tensor, ...] — each [batch_size, n_features_i].
    feature_mask: optional tuple[Tensor, ...] — groups raw features into
                  semantic units (e.g. tokens → words). Same shape as attributions.
    k_fraction: float — fraction of total (reduced) features to select.
    reduce_mode: "sum" or "weighted_sum"
        - "sum": simple sum per feature group
        - "weighted_sum": mean per group × min_group_size across all modalities
                          (penalizes groups larger than the minimum)

Output:
    Tensor of shape [batch_size, n_modalities] — fraction of top-k from each modality.

Category: characterization
"""

# ── run metric ────────────────────────────────────────────────────────────────
from torchxai.metrics.diagnosis.modality_topk_fraction import modality_topk_fraction


result = modality_topk_fraction(
    attributions=attributions,
    feature_mask=feature_mask,
    reduce_mode="sum",
)


for key, value in result.items():
    print(f"{key}: {value}")

score_modality_0: tensor([0.4239, 0.5490, 0.6934, 0.6891])
score_modality_1: tensor([0.5761, 0.4510, 0.3066, 0.3109])
